<h1>Modelagem e Machine Learning - Valores de Aluguel em São Paulo</h1>

In [23]:
# 1. verificações iniciais
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import joblib

# definir caminho do arquivo
file_path = '../data/processed/rent_clean.csv'

# ler dataset limpo
df = pd.read_csv(file_path)

# verificar as primeiras linhas
df.head()

,Price,Condo,Size,Rooms,Toilets,Suites,Parking,Elevator,Furnished,Swimming Pool,New,District,Latitude,Longitude
0,930,220,47,2,2,1,1,0,0,0,0,Artur Alvim,-23.543138,-46.479486
1,1000,148,45,2,2,1,1,0,0,0,0,Artur Alvim,-23.550239,-46.480718
2,1000,100,48,2,2,1,1,0,0,0,0,Artur Alvim,-23.542818,-46.485665
3,1000,200,48,2,2,1,1,0,0,0,0,Artur Alvim,-23.547171,-46.483014
4,1300,410,55,2,2,1,1,1,0,0,0,Artur Alvim,-23.525025,-46.482436


<p style="color: red;"><b>Nota:</p></b>Percebi que a coluna 'District' ainda é um texto. Para criação do modelo depois, irei aplicar o one-hot encoding.

In [26]:
# 2. transformar a coluna 'District' em números (One-Hot Encoding)
# aprendi: termo One-Hot Encoding
df_model = pd.get_dummies(df, columns=['District'], prefix='District')

# checar shape do novo dataframe
print(f"Novo shape: {df_model.shape}")

# verificar
df_model.head()

Novo shape: (7125, 107)


,Price,Condo,Size,Rooms,Toilets,Suites,Parking,Elevator,Furnished,Swimming Pool,...,District_Vila Jacuí,District_Vila Leopoldina,District_Vila Madalena,District_Vila Maria,District_Vila Mariana,District_Vila Matilde,District_Vila Olimpia,District_Vila Prudente,District_Vila Sônia,District_Água Rasa
0,930,220,47,2,2,1,1,0,0,0,...,False,False,False,False,False,False,False,False,False,False
1,1000,148,45,2,2,1,1,0,0,0,...,False,False,False,False,False,False,False,False,False,False
2,1000,100,48,2,2,1,1,0,0,0,...,False,False,False,False,False,False,False,False,False,False
3,1000,200,48,2,2,1,1,0,0,0,...,False,False,False,False,False,False,False,False,False,False
4,1300,410,55,2,2,1,1,1,0,0,...,False,False,False,False,False,False,False,False,False,False


In [4]:
# 3. separar features de target
X = df_model.drop('Price', axis=1)
y = df_model['Price']

print("X (features):", X.shape)
print("y (target):", y.shape)

X (features): (7125, 106)
y (target): (7125,)


In [16]:
# 4. treinar modelo
# aprendi: separar 80% pra treino e 20% pra teste
# aprendi: X_train (colunas de entrada usadas para o modelo aprender)
# aprendi: X_test (colunas de entrada que o modelo nunca viu)
# aprendi: y_train (preços correspondentes ao X_train)
# aprendi: y_test (preços reais)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

print(f"Dados para treino: {X_train.shape[0]} amostras")
print(f"Dados para teste: {X_test.shape[0]} amostras")

# treino
model = LinearRegression()

model.fit(X_train, y_train)

print("Treino finalizado.")

Dados para treino: 5700 amostras
Dados para teste: 1425 amostras
Treino finalizado.


In [17]:
# 5. prever e calcular R² score
# prever preços dos dados de teste
y_pred = model.predict(X_test)

# R² score do modelo
# aprendi: R² Score (nota do modelo, de 0 a 1)
r2_score = model.score(X_test, y_test)

print(f"R² Score: {r2_score:.4f}")

R² Score: 0.7414


In [18]:
# 6. criar dataframe para comparar preços reais e os valores previstos
compare = pd.DataFrame({
    'Preço Real': y_test.values, 
    'Previsão do Modelo': y_pred
})

compare.head(10)

,Preço Real,Previsão do Modelo
0,1700,1334.668563
1,1310,1903.280793
2,1400,2435.986623
3,1200,1230.781191
4,3400,2946.124063
5,10000,7696.427657
6,2500,1798.462903
7,3500,4645.940781
8,1300,1576.905561
9,4393,6222.729333


In [22]:
# 7. criar dataframe para salvar os nomes e pesos das colunas
coeficientes = pd.DataFrame({
    'Variável': X.columns,
    'Peso': model.coef_
})

# ordenar do maior para o menor para ver o que mais encarece o aluguel
top_caros = coeficientes.sort_values(by='Peso', ascending=False).head(10)

print("Top 10 variáveis que mais aumentam o preço:")
print(top_caros)

Top 10 variáveis que mais aumentam o preço:
                       Variável         Peso
43            District_Iguatemi  4338.896761
45          District_Itaim Bibi  3852.026001
102       District_Vila Olimpia  2906.632860
63               District_Moema  2219.270714
21            District_Brooklin  2163.529651
71           District_Pinheiros  2083.801035
52     District_Jardim Paulista  1911.949977
12   District_Alto de Pinheiros  1745.295280
26          District_Campo Belo  1404.774983
37          District_Consolação  1395.463714


In [25]:
folder_path = '../models'

# aprendi: joblib [salvar o objeto 'model' em um arquivo .pkl (pickle)]
joblib.dump(model, f'{folder_path}/modelo_rent_sp.pkl')

print("Modelo salvo.")

Modelo salvo.
